In [ ]:
# prueba indexador con una consulta de ejemplo
import pandas as pd
import sys
sys.path.append('../')

from src.indexear import IndiceInvertido

# muestra pequeña para no saturar la memoria
corpus_path = '../data/processed/corpus_limpio.csv'
df = pd.read_csv(corpus_path).fillna('').head(1000)

indexador = IndiceInvertido()
indexador.construir_desde_dataframe(df)

query = "ingeniero administracion de empresas"

tokens_consulta = indexador.nlp.tokenizar(query)
print(f"Consulta procesada (Stemming + Stopwords): {tokens_consulta}\n")


Vocabulario total: 4395 palabras únicas.
Consulta procesada (Stemming + Stopwords): ['ingenier', 'administr', 'empres']



In [7]:
for token in tokens_consulta:
    if token in indexador.indice:
        docs = indexador.indice[token] 
        print(f"El token '{token}' aparece en {len(docs)} documentos. Aquí el Top 5:")
        
        # 1. Extraemos solo los IDs de los resultados (Tomamos los primeros 5 para no saturar)
        top_doc_ids = list(docs.keys())[:5]
        
        # 2. Filtramos nuestro DataFrame original usando esos IDs
        df_resultados = df[df['job_id'].isin(top_doc_ids)].copy()
        
        # 3. Le inyectamos la "frecuencia" matemática que nos dio el índice
        df_resultados['TF'] = df_resultados['job_id'].map(docs)
        
        # 4. Ordenamos de mayor a menor frecuencia
        df_resultados = df_resultados.sort_values(by='TF', ascending=False)
        
        # 5. Elegimos SOLO las columnas legibles e importantes
        columnas_legibles = ['job_title', 'company', 'location_final', 'TF']
        # (Filtro de seguridad por si alguna columna no está en tu df temporal)
        columnas_legibles = [col for col in columnas_legibles if col in df_resultados.columns]
        
        # Imprimimos el DataFrame como una tabla bonita en el Notebook
        display(df_resultados[columnas_legibles])
        print("-" * 80) # Línea separadora
        
    else:
        print(f"El token '{token}' no se encontró en el corpus.")
        print("-" * 80)

El token 'ingenier' aparece en 15 documentos. Aquí el Top 5:


,job_title,TF
273,ingeniero de producto ingeniero de aplicaciones,4
48,tecnico o ingeniero en administracion de empresas,2
19,practica tecnico o ingeniero en administracion...,2
1,docente jp administracion de empresas tumbes,1
258,ingeniero de servicio,1


--------------------------------------------------------------------------------
El token 'administr' aparece en 1000 documentos. Aquí el Top 5:


,job_title,TF
4,ver oferta administracion de empresas auxiliar...,8
1,docente jp administracion de empresas tumbes,7
0,asistente de administracion y gerencia experie...,5
3,aprendiz universitario administracion de empresas,5
2,aprendiz universitario administracion de empre...,2


--------------------------------------------------------------------------------
El token 'empres' aparece en 1000 documentos. Aquí el Top 5:


,job_title,TF
4,ver oferta administracion de empresas auxiliar...,7
3,aprendiz universitario administracion de empresas,5
0,asistente de administracion y gerencia experie...,4
2,aprendiz universitario administracion de empre...,2
1,docente jp administracion de empresas tumbes,1


--------------------------------------------------------------------------------
